In [1]:
#python
# 커널을 "Model Serving"으로 전환한 뒤 실행합니다
import sys
print(f"Python 경로: {sys.executable}")
# 출력에 .venv가 포함되어 있으면 성공입니다
# 예: /Users/yourname/model-serving-course/.venv/bin/python

Python 경로: C:\Users\TOP\model-serving-course\.venv\Scripts\python.exe


In [3]:
%%writefile requirements.txt
# ===== Core =====
torch>=2.1.0,<2.5.0
torchvision>=0.16.0,<0.20.0

# ===== API =====
fastapi==0.115.0
uvicorn[standard]==0.30.0
pydantic>=2.0.0,<3.0.0

# ===== Frontend =====
streamlit==1.38.0

# ===== Utilities =====
requests>=2.31.0,<3.0.0
pillow>=10.0.0
python-multipart>=0.0.6

Writing requirements.txt


In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import fastapi
import streamlit

print(f"PyTorch:   {torch.__version__}")
print(f"FastAPI:   {fastapi.__version__}")
print(f"Streamlit: {streamlit.__version__}")

PyTorch:   2.12.0+cpu
FastAPI:   0.115.0
Streamlit: 1.38.0


In [3]:
!pip freeze > requirements_freeze.txt

In [4]:
import os

# 폴더 구조 생성
folders = [
    "models",          # 저장된 모델 파일 (.pth, .onnx 등)
    "app",             # FastAPI 애플리케이션 코드
    "frontend",        # Streamlit 프론트엔드 코드
    "notebooks",       # 주피터 노트북 (지금 이 파일)
    "data",            # 샘플 데이터
    "tests",           # 테스트 코드
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ {folder}/ 생성 완료")

✅ models/ 생성 완료
✅ app/ 생성 완료
✅ frontend/ 생성 완료
✅ notebooks/ 생성 완료
✅ data/ 생성 완료
✅ tests/ 생성 완료


In [5]:
%%writefile .gitignore
# 가상환경
.venv/

# Python 캐시
__pycache__/
*.pyc
*.pyo

# 모델 파일 (용량이 크므로 Git에 포함하지 않음)
models/*.pth
models/*.onnx
models/*.pt

# 주피터 체크포인트
.ipynb_checkpoints/

# OS 파일
.DS_Store
Thumbs.db
```

Writing .gitignore


In [10]:
# 1. 새 테스트 가상환경 만들기
!python -m venv .venv_test

# 2. requirements.txt로 설치
!.venv_test/Scripts/pip install -r requirements.txt -q

# 3. 버전 확인
!.venv_test/Scripts/python -c "import torch; import fastapi; print(f'torch={torch.__version__}, fastapi={fastapi.__version__}')"

'.venv_test'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'.venv_test'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.


In [11]:
import json
data = {"text": "이 영화 정말 재밌다", "return_probabilities": True}
print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "text": "이 영화 정말 재밌다",
  "return_probabilities": true
}


In [12]:
import requests
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")
print(response.status_code)   # 200
print(response.json())

200
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [13]:
import requests
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")
print(response.status_code)   # 200
print(response.json())

200
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [15]:
import shutil
shutil.rmtree('.venv_test', ignore_errors=True)
print('✅ 삭제 완료')

✅ 삭제 완료


In [16]:
response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json={
        "title": "모델 배포 테스트",
        "body": "FastAPI로 모델을 서빙합니다",
        "userId": 1
    }
)
print(f"상태 코드: {response.status_code}")   # 201
print(response.json())

상태 코드: 201
{'title': '모델 배포 테스트', 'body': 'FastAPI로 모델을 서빙합니다', 'userId': 1, 'id': 101}


In [17]:
import torch
import torch.nn as nn

In [18]:
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        ...

In [19]:
# 모델 인스턴스 생성
model = SimpleClassifier(num_classes=10)

# 더미 입력으로 동작 확인
dummy_input = torch.randn(1, 1, 28, 28)
output = model(dummy_input)

print(f"입력 크기: {dummy_input.shape}")
print(f"출력 크기: {output.shape}")   # torch.Size([1, 10])

NotImplementedError: Module [SimpleClassifier] is missing the required "forward" function

In [20]:
import torch
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 테스트
model = SimpleClassifier(num_classes=10)
dummy_input = torch.randn(1, 1, 28, 28)
output = model(dummy_input)

print(f"입력 크기: {dummy_input.shape}")
print(f"출력 크기: {output.shape}")   # torch.Size([1, 10])
print("✅ 모델 정상 동작!")

입력 크기: torch.Size([1, 1, 28, 28])
출력 크기: torch.Size([1, 10])
✅ 모델 정상 동작!


In [21]:
import os

# 모델 저장 폴더 확인
os.makedirs("models", exist_ok=True)

# eval 모드로 전환 (필수!)
model.eval()

# 저장
torch.save(model.state_dict(), "models/model_state_dict.pth")

# 파일 크기 확인
file_size = os.path.getsize("models/model_state_dict.pth")
print(f"저장 완료: models/model_state_dict.pth")
print(f"파일 크기: {file_size / 1024:.1f} KB")

저장 완료: models/model_state_dict.pth
파일 크기: 1650.9 KB


In [22]:
# 새 모델 만들고 가중치 불러오기
loaded_model = SimpleClassifier(num_classes=10)
loaded_model.load_state_dict(
    torch.load("models/model_state_dict.pth", weights_only=True)
)
loaded_model.eval()

# 동일한 입력으로 출력 비교
with torch.no_grad():
    original_output = model(dummy_input)
    loaded_output = loaded_model(dummy_input)

print(f"원본 출력:  {original_output}")
print(f"복원 출력:  {loaded_output}")
print(f"동일 여부:  {torch.allclose(original_output, loaded_output)}")  # True

원본 출력:  tensor([[-0.1036,  0.1329, -0.1258, -0.0570, -0.0139, -0.0746,  0.0775,  0.0579,
          0.0714,  0.1701]])
복원 출력:  tensor([[-0.1036,  0.1329, -0.1258, -0.0570, -0.0139, -0.0746,  0.0775,  0.0579,
          0.0714,  0.1701]])
동일 여부:  True


In [23]:
# eval 모드 확인
model.eval()

# trace 방식으로 변환
traced_model = torch.jit.trace(model, dummy_input)

# 저장
traced_model.save("models/model_traced.pt")

file_size = os.path.getsize("models/model_traced.pt")
print(f"저장 완료: models/model_traced.pt")
print(f"파일 크기: {file_size / 1024:.1f} KB")

저장 완료: models/model_traced.pt
파일 크기: 1673.2 KB


In [24]:
# 클래스 정의 없이 불러오기!
loaded_traced = torch.jit.load("models/model_traced.pt")

with torch.no_grad():
    traced_output = loaded_traced(dummy_input)

print(f"동일 여부: {torch.allclose(original_output, traced_output)}")  # True

동일 여부: True


In [25]:
!pip install onnx onnxscript onnxruntime

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ------- -------------------------------- 3.1/16.4 MB 25.8 MB/s eta 0:00:01
   ---------------------------- ----------- 11.5/16.4 MB 32.4 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 31.2 MB/s  0:00:00
   ---------------------------------------- 0.0/714.8 kB ? eta -:--:--
   ---------------------------------------- 714.8/714.8 kB 27.6 MB/s  0:00:00
   ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
   -------------------------------- ------- 10.5/13.0 MB 51.0 MB/s eta 0:00:01
   ---------------------------------------- 13.0/13.0 MB 47.6 MB/s  0:00:00

   ------ --------------------------------- 1/6 [onnxruntime]
   ------ --------------------------------- 1/6 [onnxruntime]
   ------ --------------------------------- 1/6 [onnxruntime]
   ------ --------------------------------- 1/6 [onnxruntime]
   ------ --------------------------------- 1/6 [onnxruntime]
   ------ --


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import torch, onnx

model.eval()

torch.onnx.export(
    model,
    dummy_input,
    "models/model.onnx",
    export_params=True,
    opset_version=17,
    input_names=["image"],
    output_names=["prediction"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "prediction": {0: "batch_size"},
    }
)

file_size = os.path.getsize("models/model.onnx")
print(f"저장 완료: models/model.onnx")
print(f"파일 크기: {file_size / 1024:.1f} KB")

C:\Users\TOP\AppData\Local\Temp\ipykernel_28816\2361241511.py:5: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0609 17:17:47.862000 28816 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\TOP\anaconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
저장 완료: models/model.onnx
파일 크기: 3.2 KB


In [27]:
import onnx
import onnxruntime as ort
import numpy as np

# 모델 구조 검증
onnx_model = onnx.load("models/model.onnx")
onnx.checker.check_model(onnx_model)
print("✅ ONNX 모델 검증 통과")

# ONNX Runtime으로 추론
session = ort.InferenceSession("models/model.onnx")
input_data = dummy_input.numpy()

onnx_output = session.run(
    output_names=["prediction"],
    input_feed={"image": input_data}
)

# 결과 비교
match = np.allclose(original_output.detach().numpy(), onnx_output[0], atol=1e-5)
print(f"PyTorch 출력:      {original_output.detach().numpy()}")
print(f"ONNX 출력:         {onnx_output[0]}")
print(f"동일 여부 (오차허용): {match}")  # True

✅ ONNX 모델 검증 통과
PyTorch 출력:      [[-0.10357253  0.13286574 -0.12576067 -0.05696377 -0.01394371 -0.07464406
   0.07745953  0.05794805  0.07135414  0.17006107]]
ONNX 출력:         [[-0.10357261  0.13286576 -0.12576063 -0.05696379 -0.01394378 -0.07464406
   0.07745955  0.05794808  0.07135416  0.17006099]]
동일 여부 (오차허용): True


5.1 워크플로우 미리보기  → 📖 읽기
5.2 MNIST 모델 학습     → ▶️ 코드 실행  ← 지금 여기
5.3 세 가지 방식 저장   → ▶️ 코드 실행
5.4 불러오기 & 검증     → ▶️ 코드 실행
5.5 배치 추론 테스트    → ▶️ 코드 실행
5.6 API 연결 준비       → ▶️ 코드 실행

5.2 첫 번째 셀 — import 실행
노트북에서 찾아서 실행하거나, 새 셀에 입력:

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [29]:
# 하이퍼파라미터
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 3

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

사용 디바이스: cpu


In [30]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"학습 데이터: {len(train_dataset)}개")
print(f"테스트 데이터: {len(test_dataset)}개")

100.0%
100.0%
100.0%
100.0%

학습 데이터: 60000개
테스트 데이터: 10000개


In [31]:
model = SimpleClassifier(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("✅ 모델 준비 완료")

✅ 모델 준비 완료


In [32]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")

print("✅ 학습 완료!")

Epoch 1/3 | Loss: 0.2133 | Accuracy: 93.53%
Epoch 2/3 | Loss: 0.0814 | Accuracy: 97.61%
Epoch 3/3 | Loss: 0.0631 | Accuracy: 98.13%
✅ 학습 완료!


In [33]:
import os
os.makedirs("models", exist_ok=True)

# CPU로 이동 & eval 모드
model_cpu = model.cpu()
model_cpu.eval()

# 테스트 입력 준비
test_input = test_dataset[0][0].unsqueeze(0)
test_label = test_dataset[0][1]

print(f"테스트 입력 크기: {test_input.shape}")
print(f"정답 레이블: {test_label}")

# 원본 모델 추론 결과 저장
with torch.no_grad():
    original_output = model_cpu(test_input)
    original_pred = original_output.argmax(dim=1).item()

print(f"원본 모델 예측: {original_pred}")
print(f"정답 여부: {'✅ 맞음' if original_pred == test_label else '❌ 틀림'}")

테스트 입력 크기: torch.Size([1, 1, 28, 28])
정답 레이블: 7
원본 모델 예측: 7
정답 여부: ✅ 맞음


In [34]:
torch.save(model_cpu.state_dict(), "models/mnist_state_dict.pth")
print(f"✅ state_dict: {os.path.getsize('models/mnist_state_dict.pth') / 1024:.1f} KB")

✅ state_dict: 1650.9 KB


In [35]:
traced_model = torch.jit.trace(model_cpu, test_input)
traced_model.save("models/mnist_traced.pt")
print(f"✅ TorchScript: {os.path.getsize('models/mnist_traced.pt') / 1024:.1f} KB")

✅ TorchScript: 1674.0 KB


In [36]:
torch.onnx.export(
    model_cpu, test_input, "models/mnist_model.onnx",
    export_params=True, opset_version=17,
    input_names=["image"], output_names=["prediction"],
    dynamic_axes={"image": {0: "batch_size"}, "prediction": {0: "batch_size"}}
)
print(f"✅ ONNX: {os.path.getsize('models/mnist_model.onnx') / 1024:.1f} KB")

C:\Users\TOP\AppData\Local\Temp\ipykernel_28816\657914850.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0609 17:44:16.988000 28816 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\TOP\anaconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ ONNX: 3.2 KB


In [37]:
loaded_sd = SimpleClassifier(num_classes=10)
loaded_sd.load_state_dict(
    torch.load("models/mnist_state_dict.pth", weights_only=True)
)
loaded_sd.eval()

with torch.no_grad():
    sd_output = loaded_sd(test_input)
    sd_pred = sd_output.argmax(dim=1).item()

print(f"[state_dict] 예측: {sd_pred}, 원본과 일치: {torch.allclose(original_output, sd_output)}")

[state_dict] 예측: 7, 원본과 일치: True


In [38]:
loaded_ts = torch.jit.load("models/mnist_traced.pt")

with torch.no_grad():
    ts_output = loaded_ts(test_input)
    ts_pred = ts_output.argmax(dim=1).item()

print(f"[TorchScript] 예측: {ts_pred}, 원본과 일치: {torch.allclose(original_output, ts_output)}")

[TorchScript] 예측: 7, 원본과 일치: True


In [39]:
session = ort.InferenceSession("models/mnist_model.onnx")
onnx_out = session.run(["prediction"], {"image": test_input.numpy()})
onnx_pred = np.argmax(onnx_out[0], axis=1)[0]

print(f"[ONNX] 예측: {onnx_pred}, 원본과 일치: {np.allclose(original_output.numpy(), onnx_out[0], atol=1e-5)}")

[ONNX] 예측: 7, 원본과 일치: True


In [40]:
# 테스트 데이터에서 8장을 배치로
batch_images = torch.stack([test_dataset[i][0] for i in range(8)])
batch_labels = [test_dataset[i][1] for i in range(8)]

print(f"배치 입력 크기: {batch_images.shape}")

# 세 가지 방식으로 배치 추론
with torch.no_grad():
    sd_batch = loaded_sd(batch_images).argmax(dim=1).tolist()
    ts_batch = loaded_ts(batch_images).argmax(dim=1).tolist()

onnx_batch_out = session.run(["prediction"], {"image": batch_images.numpy()})
onnx_batch = np.argmax(onnx_batch_out[0], axis=1).tolist()

# 결과 비교
print(f"\n{'이미지':<6} {'정답':<5} {'state_dict':<12} {'TorchScript':<13} {'ONNX'}")
print("-" * 50)
for i in range(8):
    match = "✅" if sd_batch[i] == ts_batch[i] == onnx_batch[i] == batch_labels[i] else "❌"
    print(f"{i:<6} {batch_labels[i]:<5} {sd_batch[i]:<12} {ts_batch[i]:<13} {onnx_batch[i]}  {match}")

배치 입력 크기: torch.Size([8, 1, 28, 28])

이미지    정답    state_dict   TorchScript   ONNX
--------------------------------------------------
0      7     7            7             7  ✅
1      2     2            2             2  ✅
2      1     1            1             1  ✅
3      0     0            0             0  ✅
4      4     4            4             4  ✅
5      1     1            1             1  ✅
6      4     4            4             4  ✅
7      9     9            9             9  ✅
